# Wan 2.2 Image-to-Video — Official Repository (T4 Colab)

Built on the **official [`Wan-Video/Wan2.2`](https://github.com/Wan-Video/Wan2.2) GitHub repository** — native inference (`generate.py`), not the `diffusers` integration. `AutoPipelineForImage2Video` is never imported anywhere in this notebook.

**Required action — one click:** `Runtime -> Restart session and run all`. Use this exact command, not plain "Run all" — it guarantees a completely clean Python process before anything runs, which is what actually avoids stale-import issues (a plain "Run all" on a runtime you've already used for something else can carry over an old package version still loaded in memory). After that one click, the only other thing you'll be asked for is the product image in Step 5.

**Python 3.13 compatibility:** the official repo's `requirements.txt` pins exact versions from when it was released, and some of those don't have a prebuilt wheel for Python 3.13 yet — installing them as pinned can fail outright on a current Colab runtime. Step 3 below installs the pinned versions first (they're what the Wan team actually tested against) and, only if that fails, automatically retries the same package list **without** version pins so `pip` resolves current releases with real Python 3.13 wheels instead. Either way, the cell finishes with a working environment or a clear error — never a silent partial install.

**Model:** `Wan-AI/Wan2.2-TI2V-5B`, run via the official `ti2v-5B` task — the only Wan 2.2 checkpoint the repo positions for a single consumer-class GPU. The 14B `T2V-A14B` / `I2V-A14B` models need **at least 80GB VRAM** per the official README, so they're not used here.

**On the output size:** the repo hard-codes which sizes each task accepts (`wan/configs/__init__.py`'s `SUPPORTED_SIZES`) — `480*832` is only valid for the 80GB+ tasks; passing it to `ti2v-5B` raises `AssertionError`, confirmed by reading that file directly. `ti2v-5B`'s only 9:16 option is `704*1280`, used below.

**Automatic T4 fallback:** the official README states TI2V-5B needs at least 24GB VRAM even with every memory-saving flag this notebook enables — a free T4 has 16GB, so a CUDA out-of-memory error is a real possibility, not a bug. Step 6 handles this itself: if generation runs out of memory, it automatically retries with a shorter clip (5s → 3s → 2s → 1s) instead of just crashing, since a free T4's exact headroom varies session to session.

**`flash_attn` is intentionally skipped**, in both install attempts. Its own attention module (`wan/modules/attention.py`) falls back to PyTorch's native `scaled_dot_product_attention` when `flash_attn` isn't installed — confirmed by reading that file directly. Compiling `flash_attn` is slow and failure-prone even outside Colab (the repo's own `INSTALL.md` calls this out); skipping it trades a little speed for an install that reliably finishes unattended.

## Step 1 — Confirm the GPU

In [ ]:
!nvidia-smi
print('\u2705 Step 1/7 done \u2014 GPU confirmed.')

## Step 2 — Clone the official Wan 2.2 repository

In [ ]:
%cd /content
!rm -rf /content/Wan2.2
!git clone --quiet https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
print('\u2705 Step 2/7 done \u2014 official Wan-Video/Wan2.2 repository cloned.')

## Step 3 — Install dependencies (self-healing for Python 3.13)

Tries the repo's exact pinned versions first; if `pip` can't find a Python-3.13-compatible wheel for one of them, automatically retries unpinned. `torch`/`torchvision`/`torchaudio` are left as Colab's own GPU-matched build in both attempts — reinstalling those is a common way to break CUDA support entirely. `flash_attn` is excluded for the reason in the notebook intro.

In [ ]:
import re
import subprocess

EXCLUDE_PREFIXES = ('flash_attn', 'torch', 'torchvision', 'torchaudio')

with open('requirements.txt') as f:
    pinned_specs = [
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
        and not any(line.strip().lower().startswith(p) for p in EXCLUDE_PREFIXES)
    ]
unpinned_specs = [re.split(r'[<>=!~\[]', spec, 1)[0].strip() for spec in pinned_specs]

def pip_install(specs, label):
    print(f'>>> {label}: pip install {" ".join(specs)}')
    return subprocess.run(['pip', 'install', '-q'] + specs, capture_output=True, text=True)

result = pip_install(pinned_specs, 'Attempt 1/2 (pinned versions from requirements.txt)')

if result.returncode != 0:
    print('\u26a0\ufe0f  Pinned install failed (likely no Python 3.13 wheel for one of them). Tail of the error:')
    print((result.stderr or result.stdout or '')[-2000:])
    print('Automatically retrying unpinned so pip can resolve current, Python-3.13-compatible releases...')
    result = pip_install(unpinned_specs, 'Attempt 2/2 (unpinned fallback)')
    if result.returncode != 0:
        print((result.stderr or result.stdout or '')[-2000:])
        raise RuntimeError(
            'Dependency install failed on both the pinned and unpinned attempts. '
            'See the pip error above for the specific package.'
        )

hf_result = subprocess.run(['pip', 'install', '-q', 'huggingface_hub[cli]'], capture_output=True, text=True)
if hf_result.returncode != 0:
    print(hf_result.stderr[-2000:])
    raise RuntimeError('Failed to install huggingface_hub[cli].')

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. Click Runtime > Change runtime type, select T4 GPU, save, '
        'then Runtime > Restart session and run all. Colab only assigns a GPU after '
        "you pick one \u2014 this is the one thing this notebook genuinely can't do for itself."
    )

try:
    import wan  # noqa: F401 — sanity check the package this whole notebook depends on actually imports
    print('wan package import: OK')
except Exception as e:
    raise RuntimeError(
        f'The wan package failed to import after installing dependencies: {e}. '
        'This usually means the unpinned fallback pulled a version combination Wan 2.2 '
        "doesn't support yet \u2014 check the error above for which package."
    )

print('\u2705 Step 3/7 done \u2014 dependencies installed and verified, no restart required from here.')

## Step 4 — Download the T4-compatible Wan 2.2 model (TI2V-5B)

Several GB from Hugging Face — first run takes a few minutes.

In [ ]:
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B
print('\u2705 Step 4/7 done \u2014 Wan-AI/Wan2.2-TI2V-5B downloaded.')

## Step 5 — Upload your product image

The only manual step in this notebook.

In [ ]:
from google.colab import files

print('Choose one product photo to upload:')
uploaded = files.upload()  # saves into the current directory: /content/Wan2.2
image_filename = next(iter(uploaded))
print(f'\u2705 Step 5/7 done \u2014 uploaded {image_filename}.')

## Step 6 — Generate a 9:16 vertical video (auto-retries on out-of-memory)

`704*1280` is `ti2v-5B`'s supported 9:16 size (see the size note above). Runs the official `generate.py` as a subprocess — if a run reports a CUDA out-of-memory error, this cell automatically shortens the clip and tries again (fresh subprocess each time, so GPU memory is fully released between attempts) instead of stopping.

In [ ]:
import subprocess

# --- Tunables -----------------------------------------------------------
SIZE = '704*1280'                       # ti2v-5B's only 9:16 option (480*832 is not valid for this task)
FPS = 24                                # fixed by the ti2v-5B model config (sample_fps)
DURATION_LADDER_SECONDS = [5, 3, 2, 1]  # automatic fallback order if one length hits CUDA OOM
PROMPT = 'A realistic, premium commercial product video. Natural, smooth camera motion, cinematic lighting.'  # edit me
SAVE_FILE = 'output.mp4'
# --------------------------------------------------------------------------

IMAGE_PATH = f'/content/Wan2.2/{image_filename}'

def frame_num_for(seconds):
    return int(round((FPS * seconds - 1) / 4)) * 4 + 1  # frame counts must be 4n+1

def run_generate(seconds):
    frame_num = frame_num_for(seconds)
    print(f'>>> Trying {frame_num} frames (~{frame_num / FPS:.1f}s @ {FPS}fps) at {SIZE}...')
    return subprocess.run(
        [
            'python', 'generate.py',
            '--task', 'ti2v-5B',
            '--size', SIZE,
            '--ckpt_dir', './Wan2.2-TI2V-5B',
            '--offload_model', 'True',
            '--convert_model_dtype',
            '--t5_cpu',
            '--image', IMAGE_PATH,
            '--prompt', PROMPT,
            '--frame_num', str(frame_num),
            '--save_file', SAVE_FILE,
        ],
        capture_output=True,
        text=True,
    )

succeeded = False
for attempt_index, seconds in enumerate(DURATION_LADDER_SECONDS):
    proc = run_generate(seconds)

    if proc.returncode == 0:
        print(f'\u2705 Step 6/7 done \u2014 generated a ~{seconds}s clip at {SIZE}: {SAVE_FILE}')
        succeeded = True
        break

    stderr_tail = (proc.stderr or '')[-4000:]
    is_oom = 'out of memory' in stderr_tail.lower()
    is_last_attempt = attempt_index == len(DURATION_LADDER_SECONDS) - 1

    if is_oom and not is_last_attempt:
        print(f'\u26a0\ufe0f  CUDA out of memory at ~{seconds}s \u2014 automatically retrying with a shorter clip...')
        continue

    print(stderr_tail)
    reason = 'CUDA out of memory even at the shortest fallback length' if is_oom else 'generate.py failed'
    raise RuntimeError(
        f'{reason}. This T4 session may simply have less free VRAM than usual right now \u2014 '
        'Runtime > Disconnect and delete runtime, then Runtime > Restart session and run all '
        'often gets you a session with more headroom (Colab does not let a notebook request '
        'a specific one). Full error above.'
    )

assert succeeded

## Step 7 — Preview and automatically download the MP4

In [ ]:
from IPython.display import Video, display
display(Video(SAVE_FILE, embed=True))

files.download("output.mp4")
print('\u2705 Step 7/7 done \u2014 output.mp4 downloaded.')